# Disorder Measures Benchmark

Benchmark `gini_impurity`, `entropy`, and `conflicts_count` across varying
distribution matrix sizes.

Each measure is called repeatedly (N_ITERATIONS) and the median time is recorded.
Results are stored per-measure in `results/dm_<name>.json` so you can run
benchmarks incrementally.

The distribution matrix is filled with random integers in `[0, MAX_COUNT)` to
simulate realistic group counts.

In [ ]:
import json
import pathlib
import statistics
import time

import numpy as np

from skrough.disorder_measures import gini_impurity, entropy, conflicts_count

In [ ]:
# Configuration
# Edit MEASURES_TO_RUN to benchmark only specific measures.
# Leave as-is to run all.

RESULTS_DIR = pathlib.Path("results")
FIGURES_DIR = pathlib.Path("figures")

# Distribution matrix sizes as (n_groups, n_values) tuples.
# Covers tiny matrices (few groups, few classes) to large ones.
MATRIX_SIZES = [
    (5, 3),
    (10, 5),
    (50, 10),
    (100, 10),
    (500, 10),
    (1000, 20),
    (2000, 20),
    (5000, 30),
    (10000, 50),
    (20000, 50),
    (50000, 100),
]

MEASURES = {
    "gini": gini_impurity,
    "entropy": entropy,
    "conflicts": conflicts_count,
}
MEASURES_TO_RUN = list(MEASURES.keys())

N_ITERATIONS = 1000  # calls per size per measure
MAX_COUNT = 100  # max value in distribution cells
RANDOM_SEED = 42

TIMEOUT_SECONDS = 60  # per measure

In [ ]:
# Benchmark runner
# Each measure's results are saved to results/dm_<name>.json.

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(RANDOM_SEED)

for measure_name in MEASURES_TO_RUN:
    measure_fn = MEASURES[measure_name]
    print(f"\n{'=' * 60}")
    print(f"Measure: {measure_name}")
    print(f"{'=' * 60}")

    measure_results = {}
    timed_out = False

    for n_groups, n_values in MATRIX_SIZES:
        if timed_out:
            print(f"  [{n_groups}x{n_values}] SKIPPED (timed out)")
            measure_results[f"{n_groups}x{n_values}"] = {
                "status": "skipped",
                "reason": "timeout",
            }
            continue

        # Generate a single distribution matrix for this size
        distribution = rng.integers(
            0, MAX_COUNT, size=(n_groups, n_values), dtype=np.int64
        )
        n_elements = int(distribution.sum())

        print(
            f"  [{n_groups}x{n_values}] Running {N_ITERATIONS} iterations...",
            end=" ",
            flush=True,
        )

        times = []
        start_wall = time.perf_counter()

        try:
            for _ in range(N_ITERATIONS):
                t0 = time.perf_counter()
                measure_fn(distribution, n_elements)
                t1 = time.perf_counter()
                times.append(t1 - t0)

            elapsed_wall = time.perf_counter() - start_wall
            median_us = statistics.median(times) * 1_000_000
            mean_us = statistics.mean(times) * 1_000_000
            print(
                f"median={median_us:.1f}us, mean={mean_us:.1f}us, wall={elapsed_wall:.2f}s"
            )

            measure_results[f"{n_groups}x{n_values}"] = {
                "status": "completed",
                "n_groups": n_groups,
                "n_values": n_values,
                "n_elements": n_elements,
                "n_iterations": N_ITERATIONS,
                "median_us": round(median_us, 3),
                "mean_us": round(mean_us, 3),
                "wall_seconds": round(elapsed_wall, 3),
            }

            if elapsed_wall > TIMEOUT_SECONDS:
                print(
                    f"  [{measure_name}] TIMEOUT ({elapsed_wall:.2f}s > {TIMEOUT_SECONDS}s)"
                )
                timed_out = True

        except Exception as e:
            elapsed_wall = time.perf_counter() - start_wall
            print(f"ERROR ({elapsed_wall:.2f}s): {e}")
            measure_results[f"{n_groups}x{n_values}"] = {
                "status": "error",
                "wall_seconds": round(elapsed_wall, 3),
                "error": str(e),
            }

    output_file = RESULTS_DIR / f"dm_{measure_name}.json"
    with open(output_file, "w") as f:
        json.dump(measure_results, f, indent=2)
    print(f"  Results saved to {output_file}")

In [ ]:
# Load all results from results/ directory

results = {}
all_measure_names = []

for result_file in sorted(RESULTS_DIR.glob("dm_*.json")):
    m_name = result_file.stem.removeprefix("dm_")
    all_measure_names.append(m_name)

    with open(result_file) as f:
        m_results = json.load(f)

    for size_str, entry in m_results.items():
        if size_str not in results:
            results[size_str] = {}
        results[size_str][m_name] = entry

all_measure_names = sorted(set(all_measure_names))
print(f"Loaded results for: {all_measure_names}")
print(f"Matrix sizes: {sorted(results.keys())}")

In [ ]:
# Summary table: median time (microseconds) by matrix size and measure

import pandas as pd

summary_rows = []
for size in sorted(results.keys()):
    for m_name in all_measure_names:
        entry = results.get(size, {}).get(m_name, {})
        summary_rows.append(
            {
                "matrix_size": size,
                "measure": m_name,
                "status": entry.get("status", "not_run"),
                "median_us": entry.get("median_us", None),
                "mean_us": entry.get("mean_us", None),
                "n_elements": entry.get("n_elements", None),
            }
        )

summary_df = pd.DataFrame(summary_rows)
pivot = summary_df.pivot(index="matrix_size", columns="measure", values="median_us")
print("\nMedian time (microseconds) by matrix size and measure:")
print(pivot.to_string())

In [ ]:
# Plot: median time vs. number of groups for each measure

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6))

for m_name in all_measure_names:
    n_groups_list = []
    times = []
    for size in sorted(results.keys()):
        entry = results.get(size, {}).get(m_name, {})
        if entry.get("status") == "completed":
            n_groups_list.append(entry["n_groups"])
            times.append(entry["median_us"])
    if n_groups_list:
        ax.plot(n_groups_list, times, marker="o", label=m_name)

ax.set_xlabel("Number of groups (rows)")
ax.set_ylabel("Median time (microseconds)")
ax.set_title("Disorder Measures Benchmark: Time vs. Matrix Size")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "disorder_measures_benchmark_plot.png", dpi=150)
plt.show()